# Google Play Store Data Analytics — Internship Dashboard

This notebook implements six visualizations over the standard Kaggle
**Google Play Store Apps** dataset.

**Required files (place next to this notebook):**
- `googleplaystore.csv`
- `googleplaystore_user_reviews.csv`  (needed for Sentiment Subjectivity in Task 5)

Download: https://www.kaggle.com/datasets/lava18/google-play-store-apps

### Documented interpretations
The brief contains a few requirements that the raw dataset cannot satisfy
literally. Decisions made (and why):

1. **Revenue** is not a column. Computed as `Price × Installs` (paid apps only;
   free apps = 0 revenue).
2. **Choropleth (Task 2)** asks for *global installs by country*, but the dataset
   has no country field. A deterministic synthetic `Country` is assigned per app
   (seeded hash) purely so the map renders. This is clearly a demo mapping, not
   real geography.
3. **Time-window gating.** Each chart only renders during its IST window. Set
   `SHOW_OVERRIDE = True` to force all charts to render for grading/review.
4. **`Size` = "Varies with device"** → treated as missing (NaN).


In [1]:
import pandas as pd, numpy as np, re, hashlib
from datetime import datetime, timezone, timedelta
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)

# Force-render all charts regardless of the IST time windows (set True for review)
SHOW_OVERRIDE = True

IST = timezone(timedelta(hours=5, minutes=30))

def in_ist_window(start_hour, end_hour):
    """True if current IST time is within [start_hour, end_hour) (24h)."""
    if SHOW_OVERRIDE:
        return True
    now = datetime.now(IST)
    h = now.hour + now.minute/60
    return start_hour <= h < end_hour

def gate(start, end, label):
    if not in_ist_window(start, end):
        print(f"[{label}] Hidden. Only visible {start}:00–{end}:00 IST. "
              f"Current IST: {datetime.now(IST):%H:%M}. Set SHOW_OVERRIDE=True to preview.")
        return False
    return True


## 1. Load & clean data

In [2]:
raw = pd.read_csv("googleplaystore.csv")
print("rows:", len(raw))

# Known bad row in the Kaggle file (shifted columns) — drop rows missing Category
df = raw.dropna(subset=["Category"]).copy()
df = df[df["Category"] != "1.9"]  # the famous corrupt row

# Rating
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

# Reviews
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")

# Installs -> int
df["Installs"] = (df["Installs"].astype(str)
                  .str.replace("[+,]", "", regex=True)
                  .replace("Free", np.nan))
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")

# Price -> float
df["Price"] = (df["Price"].astype(str).str.replace("$", "", regex=False))
df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)

# Size -> MB (k -> /1024, "Varies with device" -> NaN)
def size_to_mb(x):
    s = str(x).strip()
    if s.endswith("M"):  return float(s[:-1])
    if s.endswith("k"):  return float(s[:-1]) / 1024
    if s.endswith("G"):  return float(s[:-1]) * 1024
    return np.nan
df["Size_MB"] = df["Size"].apply(size_to_mb)

# Android version (min) -> float
def andro(x):
    m = re.match(r"(\d+\.?\d*)", str(x))
    return float(m.group(1)) if m else np.nan
df["Android_Min"] = df["Android Ver"].apply(andro)

# Last Updated -> datetime + month
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
df["Update_Month"] = df["Last Updated"].dt.month
df["Update_Period"] = df["Last Updated"].dt.to_period("M").dt.to_timestamp()

# Revenue = Price * Installs (paid only)
df["Revenue"] = np.where(df["Type"] == "Paid", df["Price"] * df["Installs"], 0)

# App name length
df["NameLen"] = df["App"].astype(str).str.len()

# de-dup apps keeping the highest review count
df = df.sort_values("Reviews", ascending=False).drop_duplicates("App").reset_index(drop=True)
print("clean rows:", len(df))
df.head(3)


rows: 10841
clean rows: 9659


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Android_Min,Update_Month,Update_Period,Revenue,NameLen
0,Facebook,SOCIAL,4.1,78158306,Varies with device,1000000000,Free,0.0,Teen,Social,2018-08-03,Varies with device,Varies with device,NaN,NaN,8,2018-08-01,0.0,8
1,WhatsApp Messenger,COMMUNICATION,4.4,69119316,Varies with device,1000000000,Free,0.0,Everyone,Communication,2018-08-03,Varies with device,Varies with device,NaN,NaN,8,2018-08-01,0.0,18
2,Instagram,SOCIAL,4.5,66577446,Varies with device,1000000000,Free,0.0,Teen,Social,2018-07-31,Varies with device,Varies with device,NaN,NaN,7,2018-07-01,0.0,9


In [3]:
# Merge user-review sentiment (subjectivity) — used by Task 5
try:
    rev = pd.read_csv("googleplaystore_user_reviews.csv")
    sent = (rev.dropna(subset=["Sentiment_Subjectivity"])
               .groupby("App")["Sentiment_Subjectivity"].mean()
               .rename("Sentiment_Subjectivity"))
    df = df.merge(sent, on="App", how="left")
    print("merged sentiment for", df["Sentiment_Subjectivity"].notna().sum(), "apps")
except FileNotFoundError:
    df["Sentiment_Subjectivity"] = np.nan
    print("user_reviews file not found — Sentiment_Subjectivity set to NaN")


merged sentiment for 816 apps


## Task 1 — Grouped bar: avg rating vs total reviews, top 10 categories by installs
Filters: apps last updated in January, app size ≥ 10M, category avg rating ≥ 4.0.
Window: **3 PM–5 PM IST**.

In [4]:
def task1():
    if not gate(15, 17, "Task 1"): return
    # apps updated in January AND size >= 10 MB
    d = df[(df["Update_Month"] == 1) & (df["Size_MB"] >= 10)].copy()
    g = d.groupby("Category").agg(
            AvgRating=("Rating","mean"),
            TotalReviews=("Reviews","sum"),
            Installs=("Installs","sum")).reset_index()
    g = g[g["AvgRating"] >= 4.0]                      # drop categories rating < 4.0
    g = g.sort_values("Installs", ascending=False).head(10)
    if g.empty:
        print("No categories pass the Task 1 filters."); return
    fig = go.Figure()
    fig.add_bar(x=g["Category"], y=g["AvgRating"], name="Avg Rating", yaxis="y1")
    fig.add_bar(x=g["Category"], y=g["TotalReviews"], name="Total Reviews", yaxis="y2")
    fig.update_layout(title="Task 1: Avg Rating & Total Reviews — Top 10 Categories by Installs",
        barmode="group", yaxis=dict(title="Avg Rating"),
        yaxis2=dict(title="Total Reviews", overlaying="y", side="right"),
        legend=dict(orientation="h"))
    fig.show()
task1()


## Task 2 — Choropleth: global installs by category (top 5)
Filters: top 5 categories by installs, categories **not** starting with A/C/G/S,
highlight categories with installs > 1,000,000. (Synthetic country mapping — see notes.)
Window: **6 PM–8 PM IST**.

In [5]:
def task2():
    if not gate(18, 20, "Task 2"): return
    d = df[~df["Category"].str[0].isin(list("ACGS"))].copy()
    top = (d.groupby("Category")["Installs"].sum()
             .sort_values(ascending=False).head(5).index.tolist())
    d = d[d["Category"].isin(top)]

    # Synthetic deterministic country assignment (dataset has no country)
    countries = ["USA","IND","BRA","GBR","DEU","FRA","JPN","RUS","CAN","AUS",
                 "MEX","IDN","NGA","ZAF","KOR"]
    def pick(app):
        h = int(hashlib.md5(str(app).encode()).hexdigest(), 16)
        return countries[h % len(countries)]
    d["Country"] = d["App"].apply(pick)

    geo = d.groupby("Country")["Installs"].sum().reset_index()
    big = (d.groupby("Category")["Installs"].sum())
    big = list(big[big > 1_000_000].index)
    fig = px.choropleth(geo, locations="Country", color="Installs",
                        locationmode="ISO-3", color_continuous_scale="Viridis",
                        title="Task 2: Global Installs by Country (Top 5 Categories, excl. A/C/G/S)")
    # visual highlight of categories exceeding 1M installs
    if big:
        fig.add_annotation(x=0.5, y=1.12, xref="paper", yref="paper", showarrow=False,
                           text="🔴 <b>Highlighted (installs > 1M):</b> " + ", ".join(big),
                           font=dict(color="crimson", size=14),
                           bgcolor="rgba(255,220,220,0.6)", bordercolor="crimson",
                           borderwidth=1, borderpad=4)
    fig.show()
task2()


## Task 3 — Dual-axis: avg installs & revenue, free vs paid, top 3 categories
Filters: installs ≥ 10,000, revenue ≥ \$10,000, Android > 4.0, size > 15M,
Content Rating = Everyone, app name ≤ 30 chars.
Window: **1 PM–2 PM IST**.

In [6]:
def task3():
    if not gate(13, 14, "Task 3"): return
    # revenue >= 10000 applied literally to every app (free apps have $0 revenue -> excluded)
    d = df[(df["Installs"] >= 10000) & (df["Revenue"] >= 10000) &
           (df["Android_Min"] > 4.0) & (df["Size_MB"] > 15) &
           (df["Content Rating"] == "Everyone") & (df["NameLen"] <= 30)].copy()
    top3 = (d.groupby("Category")["Installs"].sum()
              .sort_values(ascending=False).head(3).index.tolist())
    d = d[d["Category"].isin(top3)]
    if d.empty:
        print("No apps pass the Task 3 filters."); return
    g = d.groupby(["Category","Type"]).agg(
            AvgInstalls=("Installs","mean"),
            AvgRevenue=("Revenue","mean")).reset_index()
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for t,col in [("Free","#4C72B0"),("Paid","#DD8452")]:
        sub = g[g["Type"]==t]
        fig.add_bar(x=sub["Category"], y=sub["AvgInstalls"], name=f"{t} avg installs",
                    marker_color=col)
    for t,col in [("Free","#55A868"),("Paid","#C44E52")]:
        sub = g[g["Type"]==t]
        fig.add_trace(go.Scatter(x=sub["Category"], y=sub["AvgRevenue"], mode="lines+markers",
                      name=f"{t} avg revenue", line=dict(color=col)), secondary_y=True)
    fig.update_yaxes(title_text="Avg Installs", secondary_y=False)
    fig.update_yaxes(title_text="Avg Revenue ($)", secondary_y=True)
    fig.update_layout(title="Task 3: Avg Installs & Revenue — Free vs Paid (Top 3 Categories)",
                      barmode="group", legend=dict(orientation="h"))
    fig.show()
task3()


## Task 4 — Time series: total installs over time by category, shade >20% MoM growth
Filters: app name not starting x/y/z, category starting E/C/B, reviews > 500,
app name without letter "S". Translate Beauty→Hindi, Business→Tamil, Dating→German.
Window: **6 PM–9 PM IST**.

In [7]:
TRANS = {"BEAUTY":"सौंदर्य", "BUSINESS":"வணிகம்", "DATING":"Partnersuche"}

def task4():
    if not gate(18, 21, "Task 4"): return
    d = df[df["Category"].str[0].isin(list("ECB")) &
           (df["Reviews"] > 500) &
           ~df["App"].str[0].str.lower().isin(list("xyz")) &
           ~df["App"].str.contains("S", case=True, na=False)].copy()
    d = d.dropna(subset=["Update_Period"])
    if d.empty:
        print("No apps pass the Task 4 filters."); return
    ts = (d.groupby(["Update_Period","Category"])["Installs"].sum()
            .reset_index().sort_values("Update_Period"))
    fig = go.Figure()
    for cat in ts["Category"].unique():
        sub = ts[ts["Category"]==cat].copy()
        sub["MoM"] = sub["Installs"].pct_change()
        label = TRANS.get(cat, cat)
        fig.add_trace(go.Scatter(x=sub["Update_Period"], y=sub["Installs"],
                      mode="lines+markers", name=label))
        # shade segments with >20% MoM growth
        for i in range(1, len(sub)):
            if sub["MoM"].iloc[i] > 0.20:
                fig.add_trace(go.Scatter(
                    x=[sub["Update_Period"].iloc[i-1], sub["Update_Period"].iloc[i]],
                    y=[sub["Installs"].iloc[i-1], sub["Installs"].iloc[i]],
                    fill="tozeroy", mode="none", showlegend=False,
                    fillcolor="rgba(255,165,0,0.25)"))
    fig.update_layout(title="Task 4: Installs Over Time by Category (>20% MoM shaded)",
                      xaxis_title="Month", yaxis_title="Total Installs")
    fig.show()
task4()


## Task 5 — Bubble: size (MB) vs rating, bubble = installs
Filters: rating > 3.5; categories Game/Beauty/Business/Comics/Communication/Dating/
Entertainment/Social/Events; reviews > 500; name without "S"; subjectivity > 0.5;
installs > 50k. Highlight Game in pink. Translate Beauty→Hindi, Business→Tamil, Dating→German.
Window: **5 PM–7 PM IST**.

In [8]:
def task5():
    if not gate(17, 19, "Task 5"): return
    cats = ["GAME","BEAUTY","BUSINESS","COMICS","COMMUNICATION","DATING",
            "ENTERTAINMENT","SOCIAL","EVENTS"]
    d = df[df["Category"].isin(cats) & (df["Rating"] > 3.5) &
           (df["Reviews"] > 500) & (df["Installs"] > 50000) &
           ~df["App"].str.contains("S", case=True, na=False) &
           (df["Sentiment_Subjectivity"] > 0.5)].copy()
    if d.empty:
        print("No apps pass the Task 5 filters (often because sentiment file is unmerged)."); return
    d["Label"] = d["Category"].replace(TRANS)
    color_map = {TRANS.get("BEAUTY","BEAUTY"):"#888"}  # default handled by px
    fig = px.scatter(d, x="Size_MB", y="Rating", size="Installs", color="Label",
                     hover_name="App", size_max=50,
                     title="Task 5: Size vs Rating (bubble = installs)")
    # highlight Game in pink
    fig.for_each_trace(lambda t: t.update(marker_color="hotpink") if t.name=="GAME" else None)
    fig.show()
task5()


## Task 6 — Stacked area: cumulative installs over time by category
Filters: rating ≥ 4.2, name without digits, category starting T/P, reviews > 1000,
size 20–80 MB. Legend: Travel & Local→French, Productivity→Spanish, Photography→Japanese.
Intensify color where any category MoM growth > 25%.
Window: **4 PM–6 PM IST**.

In [9]:
TRANS6 = {"TRAVEL_AND_LOCAL":"Voyages et local", "PRODUCTIVITY":"Productividad",
          "PHOTOGRAPHY":"写真"}

def task6():
    if not gate(16, 18, "Task 6"): return
    d = df[df["Category"].str[0].isin(list("TP")) & (df["Rating"] >= 4.2) &
           (df["Reviews"] > 1000) &
           (df["Size_MB"].between(20, 80)) &
           ~df["App"].str.contains(r"\d", regex=True, na=False)].copy()
    d = d.dropna(subset=["Update_Period"])
    if d.empty:
        print("No apps pass the Task 6 filters."); return
    ts = (d.groupby(["Update_Period","Category"])["Installs"].sum()
            .reset_index().sort_values("Update_Period"))
    ts["Cumulative"] = ts.groupby("Category")["Installs"].cumsum()
    ts["Label"] = ts["Category"].replace(TRANS6)
    fig = px.area(ts, x="Update_Period", y="Cumulative", color="Label",
                  title="Task 6: Cumulative Installs Over Time by Category")
    # color-intensity highlight: shade months where any category grew >25% MoM
    hot = set()
    for cat in ts["Category"].unique():
        sub = ts[ts["Category"]==cat].sort_values("Update_Period")
        mom = sub["Installs"].pct_change()
        hot.update(sub.loc[mom > 0.25, "Update_Period"].tolist())
    periods = pd.to_datetime(sorted(pd.Series(ts["Update_Period"].unique())))
    half = (periods[1]-periods[0]) / 2 if len(periods) > 1 else pd.Timedelta(days=15)
    for m in sorted(hot):
        m = pd.Timestamp(m)
        fig.add_vrect(x0=m-half, x1=m+half, fillcolor="rgba(0,0,0,0.28)",
                      line_width=0, layer="above")
    if hot:
        fig.update_layout(title="Task 6: Cumulative Installs Over Time — dark bands = "
                                ">25% MoM growth months")
    fig.show()
task6()
